# Assignment 11: Fine-Tuning GPT-2 on a Custom Text Style

## Objective

Understand how fine-tuning helps a pretrained GPT-2 language model adapt to a specific writing style.

## Workflow

1. Create/load a 200–300-line custom travel-story dataset.
2. Clean the dataset.
3. Load original GPT-2 and generate baseline text using the selected travel-story prompt.
4. Tokenize the travel-story dataset.
5. Fine-tune GPT-2 using Hugging Face Trainer.
6. Generate text using the fine-tuned model.
7. Compare both outputs and save the required files.

**Style chosen:** original fictional travel-story writing.

In [1]:
!pip install -q transformers accelerate

# 2. Import libraries

In [ ]:
import os
import re
import random
import torch
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
GPU available: False


# 3. Load the included dataset

In [3]:
from google.colab import files

uploaded = files.upload()

Saving travel_stories_dataset.txt to travel_stories_dataset.txt


In [5]:
dataset_file = "travel_stories_dataset.txt"

with open(dataset_file, "r", encoding="utf-8") as f:
    dataset_lines = f.read().splitlines()

print("Dataset lines:", len(dataset_lines))

print("\nFirst five examples:")
for line in dataset_lines[:5]:
    print(line)

Dataset lines: 183

First five examples:
A traveler arrived at a quiet coastal village just before sunrise and found the streets covered in soft golden light.
She carried a small backpack, a notebook, and a camera that had belonged to her grandfather.
The village stood between green hills and a wide blue sea.
Fishing boats moved slowly across the water while birds circled above the harbor.
She walked toward the old lighthouse that appeared on the map she had received before the journey.


# Clean Text

In [6]:
cleaned_lines = []

for line in dataset_lines:
    line = line.strip()
    line = re.sub(r"\s+", " ", line).strip()

    if line:
        cleaned_lines.append(line)

cleaned_lines = list(dict.fromkeys(cleaned_lines))

with open("cleaned_travel_dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(cleaned_lines))

print("Before cleaning:", len(dataset_lines))
print("After cleaning:", len(cleaned_lines))

Before cleaning: 183
After cleaning: 183


# 5. Load GPT-2

In [7]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(model_name)
base_model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = base_model.to(device)

print("Original GPT-2 loaded on:", device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Original GPT-2 loaded on: cpu


# 6. Generate BASE model output

In [8]:
prompt = "The traveler arrived at the small village and"

In [9]:
def generate_text(model, prompt, seed=42):

    model.eval()

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


base_output = generate_text(base_model, prompt)

print("========== BASE MODEL OUTPUT ==========")
print(base_output)

with open(
    "base_model_output.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(base_output)

========== BASE MODEL OUTPUT ==========
The traveler arrived at the small village and made his way to a house in which he sat on an iron stool, then took out one of its bottles for himself. He looked up from it as if looking back into that bottle with sadness; but when all was said and done there were no more books left after him—a man who had lost so many friends by this means!
"But I shall not give you any further encouragement,"


# 7. Prepare dataset

In [10]:
from torch.utils.data import Dataset

class TravelDataset(Dataset):

    def __init__(
        self,
        lines,
        tokenizer,
        max_length=96
    ):

        self.examples = []

        for line in lines:

            encoded = tokenizer(
                line,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt"
            )

            self.examples.append({
                "input_ids":
                    encoded["input_ids"].squeeze(0),

                "attention_mask":
                    encoded["attention_mask"].squeeze(0)
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


train_dataset = TravelDataset(
    cleaned_lines,
    tokenizer
)

print(
    "Training examples:",
    len(train_dataset)
)

Training examples: 183


# 8. Fine-tune GPT-2

In [11]:
training_args = TrainingArguments(
    output_dir="./fine_tuned_gpt2_travel",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=2,

    learning_rate=5e-5,

    logging_steps=20,

    save_strategy="epoch",

    report_to="none",

    fp16=torch.cuda.is_available()
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=base_model,

    args=training_args,

    train_dataset=train_dataset,

    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
20,3.563188
40,3.313498
60,2.687236
80,2.622960
100,2.353844
120,2.255163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=138, training_loss=2.7180599820786626, metrics={'train_runtime': 940.4067, 'train_samples_per_second': 0.584, 'train_steps_per_second': 0.147, 'total_flos': 26896748544000.0, 'train_loss': 2.7180599820786626, 'epoch': 3.0})

# 9. Save fine-tuned model

In [12]:
output_dir = "./fine_tuned_gpt2_travel"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Fine-tuned model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved!


# 10. Generate AFTER output

In [13]:
fine_tuned_model = GPT2LMHeadModel.from_pretrained(
    output_dir
)

fine_tuned_model = fine_tuned_model.to(device)

fine_tuned_output = generate_text(
    fine_tuned_model,
    prompt
)

print("========== FINE-TUNED MODEL OUTPUT ==========")
print(fine_tuned_output)

with open(
    "fine_tuned_model_output.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(fine_tuned_output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

========== FINE-TUNED MODEL OUTPUT ==========
The traveler arrived at the small village and watched a river flowing toward her.
"When I entered town, there was no other place for me. The mountains were peaceful but filled with birds singing softly across the evening air."
After returning to civilization after completing several years of traveling in different regions she moved on into another city where travelers continued their journey until sunrise.


In [15]:

print("=" * 60)
print("PROMPT")
print(prompt)
print("\n" + "=" * 60)
print("ORIGINAL GPT-2")
print(base_output)
print("\n" + "=" * 60)
print("FINE-TUNED GPT-2")
print(fine_tuned_output)

PROMPT
The traveler arrived at the small village and

ORIGINAL GPT-2
The traveler arrived at the small village and made his way to a house in which he sat on an iron stool, then took out one of its bottles for himself. He looked up from it as if looking back into that bottle with sadness; but when all was said and done there were no more books left after him—a man who had lost so many friends by this means!
"But I shall not give you any further encouragement,"

FINE-TUNED GPT-2
The traveler arrived at the small village and watched a river flowing toward her.
"When I entered town, there was no other place for me. The mountains were peaceful but filled with birds singing softly across the evening air."
After returning to civilization after completing several years of traveling in different regions she moved on into another city where travelers continued their journey until sunrise.


# 11. Before vs After comparison

In [14]:
print("========== BEFORE FINE-TUNING ==========")
print(base_output)

print("\n========== AFTER FINE-TUNING ==========")
print(fine_tuned_output)

========== BEFORE FINE-TUNING ==========
The traveler arrived at the small village and made his way to a house in which he sat on an iron stool, then took out one of its bottles for himself. He looked up from it as if looking back into that bottle with sadness; but when all was said and done there were no more books left after him—a man who had lost so many friends by this means!
"But I shall not give you any further encouragement,"

========== AFTER FINE-TUNING ==========
The traveler arrived at the small village and watched a river flowing toward her.
"When I entered town, there was no other place for me. The mountains were peaceful but filled with birds singing softly across the evening air."
After returning to civilization after completing several years of traveling in different regions she moved on into another city where travelers continued their journey until sunrise.


# Before and After Comparison

## Prompt

**The traveler arrived at the small village and**

| Before Fine-Tuning (Base GPT-2) | After Fine-Tuning (Travel Story GPT-2) |
|---|---|
| Generated a general continuation based on the original GPT-2 training. | Generated a continuation more related to travel stories. |
| The output may contain unrelated or general content. | The output contains travel-related words and situations. |
| Less focused on a specific writing style. | More focused on travelers, villages, journeys, roads, mountains, and destinations. |
| General language-generation behavior. | Adapted toward the style of the custom travel-story dataset. |

## Conclusion

Fine-tuning helped GPT-2 adapt to the writing style of the custom
travel-story dataset and produce text that is more relevant to
travel-related storytelling.